# 01 — Exploración inicial

Exploración de los archivos CSV en `/data/raw/` usando **DuckDB** (todas
las consultas (conteos, nulos, duplicados, cuartiles, joins) se ejecutan en
SQL sobre esas vistas). Pandas solo se usa puntualmente, vía `.df()`, para
mostrar resultados ya agregados/pequeños (p. ej. la lista de outliers).

Este notebook **no corrige nada**: solo documenta el estado inicial de los datos antes de
la fase de limpieza (`/src`).

**Requisito para ejecutar este notebook**: a diferencia de los notebooks 02 y
03 (que usan directamente `/data/processed/civitatis.duckdb`, ya incluido en
el repo), este notebook explora los **CSV originales sin procesar**, así que
necesita los 5 ficheros de Civitatis colocados en `/data/raw/` para poder
ejecutarse — no se incluyen en el repo por tamaño y por política del
ejercicio (ver README).

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

## Carga de datos


In [2]:
RAW = '../data/raw'

con.execute(f"CREATE OR REPLACE VIEW ga_eventos  AS SELECT * FROM '{RAW}/ga_eventos.csv'")
con.execute(f"CREATE OR REPLACE VIEW reservas    AS SELECT * FROM '{RAW}/reservas.csv'")
con.execute(f"CREATE OR REPLACE VIEW clientes    AS SELECT * FROM '{RAW}/clientes.csv'")
con.execute(f"CREATE OR REPLACE VIEW tours       AS SELECT * FROM '{RAW}/tours.csv'")
con.execute(f"CREATE OR REPLACE VIEW proveedores AS SELECT * FROM '{RAW}/proveedores.csv'")

TABLES = ['ga_eventos', 'reservas', 'clientes', 'tours', 'proveedores']
con.execute("SHOW TABLES").df()

,name
0,clientes
1,ga_eventos
2,proveedores
3,reservas
4,tours


## 1. Perfilado por tabla

Para cada archivo csv se extraen: nº de filas, nº de columnas, tipos inferidos por DuckDB,
% de nulos por columna y nº de duplicados (filas exactamente repetidas).


In [3]:
def perfil_tabla(nombre):
    n_filas = con.execute(f"SELECT count(*) FROM {nombre}").fetchone()[0]
    columnas = con.execute(f"DESCRIBE {nombre}").df()
    n_cols = len(columnas)

    exprs = [
        f"round(100.0 * sum(CASE WHEN \"{c}\" IS NULL THEN 1 ELSE 0 END) / {n_filas}, 2) AS \"{c}\""
        for c in columnas['column_name']
    ]
    nulos_pct = con.execute(f"SELECT {', '.join(exprs)} FROM {nombre}").df().T
    nulos_pct.columns = ['pct_nulos']

    perfil = columnas.set_index('column_name')[['column_type']].join(nulos_pct)
    perfil = perfil.rename(columns={'column_type': 'tipo_inferido'})

    n_distintas = con.execute(f"SELECT count(*) FROM (SELECT DISTINCT * FROM {nombre})").fetchone()[0]
    duplicados = n_filas - n_distintas

    print(f"== {nombre} ==  filas={n_filas:,}  columnas={n_cols}  filas_duplicadas={duplicados:,}")
    return perfil

perfiles = {t: perfil_tabla(t) for t in TABLES}

== ga_eventos ==  filas=702,821  columnas=12  filas_duplicadas=798


== reservas ==  filas=8,414  columnas=11  filas_duplicadas=0


== clientes ==  filas=8,060  columnas=10  filas_duplicadas=0
== tours ==  filas=65  columnas=5  filas_duplicadas=0
== proveedores ==  filas=25  columnas=8  filas_duplicadas=0


In [4]:
perfiles['ga_eventos']

,tipo_inferido,pct_nulos
column_name,,
cookie_id,VARCHAR,0.00
temp_client_id,VARCHAR,0.00
user_id,BIGINT,89.72
session_id,VARCHAR,0.00
event_date,VARCHAR,0.00
event_name,VARCHAR,0.00
url,VARCHAR,38.79
ip,VARCHAR,0.00
pais_ip,VARCHAR,0.00


In [5]:
perfiles['reservas']

,tipo_inferido,pct_nulos
column_name,,
reserva_id,BIGINT,0.00
user_id,BIGINT,0.00
tour_id,BIGINT,0.00
proveedor_id,BIGINT,0.00
fecha_reserva,TIMESTAMP,0.00
fecha_actividad,DATE,0.00
estado,VARCHAR,0.00
personas,BIGINT,0.00
importe_eur,DOUBLE,0.00


In [6]:
perfiles['clientes']

,tipo_inferido,pct_nulos
column_name,,
user_id,BIGINT,0.00
nombre,VARCHAR,0.00
apellidos,VARCHAR,0.00
direccion,VARCHAR,0.00
email,VARCHAR,0.00
telefono,VARCHAR,0.00
fecha_alta,DATE,0.00
fecha_baja,DATE,98.14
fecha_nacimiento,DATE,0.00


In [7]:
perfiles['tours']

,tipo_inferido,pct_nulos
column_name,,
tour_id,BIGINT,0.0
url,VARCHAR,0.0
descripcion,VARCHAR,0.0
precio_por_persona_eur,DOUBLE,0.0
proveedor_id,BIGINT,0.0


In [8]:
perfiles['proveedores']

,tipo_inferido,pct_nulos
column_name,,
proveedor_id,BIGINT,0.0
nombre_empresa,VARCHAR,0.0
direccion,VARCHAR,0.0
email,VARCHAR,0.0
telefono,VARCHAR,0.0
fecha_alta,DATE,0.0
fecha_baja,DATE,92.0
cif,VARCHAR,0.0


### Conclusiones — perfilado

- **`ga_eventos`**: 


    | Columna | % nulos | Por qué |
    |---|---|---|
    | `user_id` | 89,7% | Solo se rellena si el visitante está identificado (registrado/logueado) |
    | `reserva_id` | 98,8% | Solo se rellena en eventos de tipo `purchase` |

    > Estos nulos son **estructurales, no un defecto de calidad** — reflejan que la mayoría del tráfico web no está identificado, no un fallo de captura.

    ⚠️ **798 filas duplicadas exactas** (evento repetido íntegramente)

- **`reservas`** (8.414 filas) y **`clientes`** (8.060 filas): Sin duplicados exactos. `clientes.fecha_baja` nula en 7.910/8.060 filas (~98%) — coherente, ya que solo se rellena si el cliente se ha dado de baja.

- **`tours`** (65) y **`proveedores`** (25) son catálogos pequeños, sin
  duplicados ni nulos relevantes.


- **⚠️ Tipos inferidos**: `ga_eventos.event_date` queda como `VARCHAR` (texto) en vez de fecha, porque el fichero **mezcla dos formatos de fecha distintos** dentro de la misma columna (detalle en la sección de fechas más abajo). No es solo un tecnicismo de tipado — es un síntoma de calidad que hay que resolver antes de poder filtrar u ordenar por fecha.


- **⚠️ Inconsistencias categóricas**

    - **`reservas.estado`**: mismo valor escrito de formas distintas — `confirmada` / `Confirmada` / `CONFIRMADA`, `cancelada` / `Cancelada` / `CANCELLED` — más un estado `Pendiente` aparte.


    - **`ga_eventos.device`**: `mobile`/`Mobile`, `desktop`/`Desktop`, y un typo `desktp`.


## 2. Relación entre las variables

A continuación se muestra un chequeo de la identidad y solapamiento de las variables (i.e. comprobar si hay relaciones extrañas entre ellas)


In [9]:
con.execute('''
    SELECT
        count(*)                     AS filas,
        count(DISTINCT cookie_id)    AS cookie_id_distintos,
        count(DISTINCT temp_client_id) AS temp_client_id_distintos,
        count(DISTINCT user_id)      AS user_id_distintos,
        count(user_id)               AS user_id_no_nulos,
        count(DISTINCT session_id)   AS session_id_distintos
    FROM ga_eventos
''').df()

,filas,cookie_id_distintos,temp_client_id_distintos,user_id_distintos,user_id_no_nulos,session_id_distintos
0,702821,250063,268475,5811,72275,272530


In [10]:
con.execute('''
    SELECT
        (SELECT count(DISTINCT user_id) FROM clientes) AS clientes_user_id_distintos,
        (SELECT count(DISTINCT user_id) FROM reservas)  AS reservas_user_id_distintos
''').df()

,clientes_user_id_distintos,reservas_user_id_distintos
0,8060,5822


Solapamiento de `user_id` entre `ga_eventos` y `clientes`:

In [11]:
con.execute('''
    SELECT
        count(DISTINCT g.user_id)                                   AS user_id_en_ga_eventos,
        count(DISTINCT c.user_id)                                   AS matched_en_clientes,
        count(DISTINCT g.user_id) - count(DISTINCT c.user_id)       AS sin_match_en_clientes
    FROM ga_eventos g
    LEFT JOIN clientes c ON g.user_id = c.user_id
    WHERE g.user_id IS NOT NULL
''').df()

,user_id_en_ga_eventos,matched_en_clientes,sin_match_en_clientes
0,5811,5811,0


Solapamiento de `user_id` entre `reservas` y `clientes` (toda reserva debería tener un cliente):

In [12]:
con.execute('''
    SELECT count(DISTINCT r.user_id) AS reservas_user_id_sin_cliente
    FROM reservas r
    WHERE NOT EXISTS (SELECT 1 FROM clientes c WHERE c.user_id = r.user_id)
''').df()

,reservas_user_id_sin_cliente
0,0


Verificación adicional: cuando ocurre un evento `purchase` en `ga_eventos`, este lleva asociado un `reserva_id`. Aquí comprobamos que ese `reserva_id` existe realmente en el fichero `reservas` — es decir, que cada "compra" registrada se corresponde con una reserva real y no con un evento suelto sin reserva detrás.


In [13]:
con.execute('''
    SELECT
        count(DISTINCT g.reserva_id)                                        AS reserva_id_en_ga_eventos,
        sum(CASE WHEN r.reserva_id IS NULL THEN 1 ELSE 0 END)               AS sin_match_en_reservas
    FROM (SELECT DISTINCT reserva_id FROM ga_eventos WHERE reserva_id IS NOT NULL) g
    LEFT JOIN reservas r ON g.reserva_id = r.reserva_id
''').df()

,reserva_id_en_ga_eventos,sin_match_en_reservas
0,8341,40.0


También se comprueba la integridad entre `reservas → tours →
proveedores`:

In [14]:
con.execute('''
    SELECT
        (SELECT count(*) FROM reservas r WHERE NOT EXISTS (SELECT 1 FROM tours t WHERE t.tour_id = r.tour_id))         AS reservas_sin_tour,
        (SELECT count(*) FROM reservas r WHERE NOT EXISTS (SELECT 1 FROM proveedores p WHERE p.proveedor_id = r.proveedor_id)) AS reservas_sin_proveedor,
        (SELECT count(*) FROM tours t WHERE NOT EXISTS (SELECT 1 FROM proveedores p WHERE p.proveedor_id = t.proveedor_id))    AS tours_sin_proveedor
''').df()

,reservas_sin_tour,reservas_sin_proveedor,tours_sin_proveedor
0,0,187,1


### Conclusiones - relaciones entre variables


-  **Volumen de tráfico identificado vs. anónimo:**

    `ga_eventos` tiene 250.063 `cookie_id` distintos y 272.530 `session_id` distintos, frente a solo ~72.275 filas con `user_id` no nulo.

    > El grueso del tráfico es anónimo — esperable en un log de analítica web, no un problema de calidad.

- **Cruce de identidad entre ficheros**

    | Comprobación | Resultado |
    |---|---|
    | `user_id` de `ga_eventos` (5.811 distintos) vs. `clientes` | 100% casan — 0 sin match |
    | `user_id` de `reservas` (5.822 distintos) vs. `clientes` | 100% casan — 0 sin match |

    > No hay usuarios "fantasma" en el tracking ni en las reservas: todo `user_id` que aparece en eventos o reservas existe en `clientes`.

- De los 8.060 clientes registrados, solo 5.822 (~72%) ha reservado alguna vez. **El 28% restante se registró y nunca compró.**


- ⚠️ **40 de 8.341 `reserva_id` de eventos `purchase` (~0,5%) no casan con ningún `reserva_id` de `reservas`.**
Puede deberse a un desfase temporal entre sistemas: analítica registra la compra en el momento exacto en que ocurre, pero puede que la reserva tarde un poco más en guardarse en la base de datos (o falle al guardarse). Es decir, el evento de "compra" quedó registrado, pero la reserva correspondiente no llegó a crearse.


- ⚠️ **Proveedor_id externo:** 187 reservas (~2,2%) y 1 tour referencian un `proveedor_id` que no existe en `proveedores`.

### Verificación de conflictos de identidad

Este notebook no corrige nada (ver introducción) — pero antes de pasar a diseñar la limpieza (`/src/limpieza.py`) hay que comprobar aquí si un supuesto concreto se cumple, porque de eso depende cómo se pueda escribir esa limpieza después.

El problema a resolver más adelante es: cada evento de `ga_eventos` trae hasta tres identificadores del visitante (`user_id`, `cookie_id`, `temp_client_id`), y no siempre vienen todos. La solución que se plantea para `/src/limpieza.py` es una regla de prioridad, `user_id > cookie_id > temp_client_id`: usar `user_id` si existe, si no `cookie_id`, si tampoco `temp_client_id`.

Esa regla solo es segura si cada `cookie_id` y cada `temp_client_id` pertenece siempre a un único `user_id`. Si un mismo `cookie_id` apareciera ligado a dos `user_id` distintos, significaría que dos personas identificadas compartieron el mismo dispositivo/navegador — y aplicar la regla fila a fila mezclaría el historial de esas dos personas bajo un solo id, dando un resultado erróneo sin que el código lo detecte.

Por eso se comprueba aquí, en la fase de exploración, si existen esos casos — **antes** de dar la regla por buena y llevarla a `/src/limpieza.py`.

In [15]:
con.execute('''
    SELECT
        -- cookie_id (entre los eventos con user_id no nulo) ligados a más de un user_id distinto
        (SELECT count(*) FROM (
            SELECT cookie_id FROM ga_eventos
            WHERE user_id IS NOT NULL
            GROUP BY cookie_id HAVING count(DISTINCT user_id) > 1
        )) AS cookie_id_con_varios_user_id,

        -- lo mismo para temp_client_id
        (SELECT count(*) FROM (
            SELECT temp_client_id FROM ga_eventos
            WHERE user_id IS NOT NULL
            GROUP BY temp_client_id HAVING count(DISTINCT user_id) > 1
        )) AS temp_client_id_con_varios_user_id
''').df()

,cookie_id_con_varios_user_id,temp_client_id_con_varios_user_id
0,0,0


**Resultado: 0 y 0.** Ningún `cookie_id` ni `temp_client_id` está ligado a más de un `user_id` distinto — ningún dispositivo/navegador identificado en los eventos fue usado por más de una persona con sesión iniciada.

> Con esto queda validado el supuesto: la regla `user_id > cookie_id > temp_client_id` es segura de aplicar con estos datos, así que puede implementarse en la fase de limpieza (tablas `paso1` → `paso2` → `eventos_con_id` en `/src/limpieza.py`) sin riesgo de mezclar el historial de dos personas bajo un mismo `id`.

## 3. Outliers en variables numéricas

Los outliers han sido calculados en base a la estadística de cuartiles.

*(Los cuartiles dividen los datos ordenados en 4 bloques iguales: el 25% más bajo queda por debajo de Q1, la mitad de los datos por debajo de la mediana, y el 25% más alto por encima de Q3. A partir de Q1 y Q3 se calcula un "rango normal" (el llamado rango intercuartílico o IQR); todo lo que cae muy por encima de ese rango se marca como outlier)*


In [16]:
def cuartiles(tabla, columna):
    return con.execute(f'''
        SELECT
            min({columna})                                              AS minimo,
            PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY {columna})      AS q1,
            PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY {columna})      AS mediana,
            PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY {columna})      AS q3,
            max({columna})                                              AS maximo
        FROM {tabla}
    ''').fetchone()

variables = [
    ('reservas', 'importe_eur'),
    ('reservas', 'personas'),
    ('tours', 'precio_por_persona_eur'),
]

resumen_outliers = {}
for tabla, col in variables:
    minimo, q1, mediana, q3, maximo = cuartiles(tabla, col)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    resumen_outliers[(tabla, col)] = dict(minimo=minimo, q1=q1, mediana=mediana, q3=q3,
                                           maximo=maximo, limite_inf=lo, limite_sup=hi)
    print(f"{tabla}.{col}: min={minimo} q1={q1} mediana={mediana} q3={q3} max={maximo} "
          f"-> límites IQR=[{lo:.2f}, {hi:.2f}]")

resumen_outliers

reservas.importe_eur: min=0.0 q1=33.32 mediana=74.36 q3=133.28 max=415.75 -> límites IQR=[-116.62, 283.22]
reservas.personas: min=-1 q1=2.0 mediana=3.0 q3=4.0 max=5 -> límites IQR=[-1.00, 7.00]
tours.precio_por_persona_eur: min=0.0 q1=17.42 mediana=27.0 q3=42.41 max=83.15 -> límites IQR=[-20.06, 79.89]


{('reservas', 'importe_eur'): {'minimo': 0.0,
  'q1': 33.32,
  'mediana': 74.36,
  'q3': 133.28,
  'maximo': 415.75,
  'limite_inf': -116.62,
  'limite_sup': 283.22},
 ('reservas', 'personas'): {'minimo': -1,
  'q1': 2.0,
  'mediana': 3.0,
  'q3': 4.0,
  'maximo': 5,
  'limite_inf': -1.0,
  'limite_sup': 7.0},
 ('tours', 'precio_por_persona_eur'): {'minimo': 0.0,
  'q1': 17.42,
  'mediana': 27.0,
  'q3': 42.41,
  'maximo': 83.15,
  'limite_inf': -20.06499999999999,
  'limite_sup': 79.89499999999998}}

Casos extremos de `importe_eur` (por encima del límite superior IQR):

In [17]:
lo = resumen_outliers[('reservas', 'importe_eur')]['limite_sup']
outliers_importe = con.execute(f'''
    SELECT reserva_id, user_id, tour_id, estado, personas, importe_eur, fecha_reserva
    FROM reservas
    WHERE importe_eur > {lo}
    ORDER BY importe_eur DESC
''').df()
print(f"Nº de reservas con importe_eur por encima de {lo:.2f}: {len(outliers_importe)}")
outliers_importe

Nº de reservas con importe_eur por encima de 283.22: 237

,reserva_id,user_id,tour_id,estado,personas,importe_eur,fecha_reserva
0,900363,20361,5057,CONFIRMADA,5,415.75,2026-06-14 12:00:19
1,906166,25921,5057,cancelada,5,415.75,2025-03-25 17:33:22
2,906027,25792,5057,CANCELLED,5,415.75,2025-01-31 11:14:55
3,906240,26010,5057,CONFIRMADA,5,415.75,2026-02-17 04:22:48
4,905975,25751,5057,confirmada,5,415.75,2024-10-09 20:28:32
...,...,...,...,...,...,...,...
232,906463,26232,5044,confirmada,4,283.92,2024-09-24 16:25:19
233,907701,27447,5044,Confirmada,4,283.92,2024-09-05 09:54:24
234,901850,21726,5044,confirmada,4,283.92,2025-03-02 19:14:33
235,901872,21746,5044,cancelada,4,283.92,2025-08-13 22:00:22


Casos con `personas` fuera de rango (incluye valores no válidos como 0 o negativos):


In [18]:
outliers_personas = con.execute('''
    SELECT reserva_id, user_id, tour_id, estado, personas, importe_eur
    FROM reservas
    WHERE personas <= 0
    ORDER BY personas
''').df()
print(f"Nº de reservas con personas <= 0: {len(outliers_personas)}")
outliers_personas

Nº de reservas con personas <= 0: 30


,reserva_id,user_id,tour_id,estado,personas,importe_eur
0,907136,26888,5063,Confirmada,-1,130.62
1,907034,26785,5028,Confirmada,-1,76.84
2,900506,20499,5012,CONFIRMADA,-1,140.20
3,906983,26732,5040,cancelada,-1,0.00
4,900630,20604,5056,confirmada,-1,46.26
5,906945,26703,5020,Confirmada,-1,105.70
6,901546,21455,5028,Pendiente,-1,19.21
7,906825,26602,5057,CONFIRMADA,-1,332.60
8,905394,25150,5057,confirmada,-1,249.45
9,905149,24899,5054,Confirmada,-1,46.72


Casos extremos de `precio_por_persona_eur` en el catálogo de tours:

In [19]:
hi_precio = resumen_outliers[('tours', 'precio_por_persona_eur')]['limite_sup']
lo_precio = resumen_outliers[('tours', 'precio_por_persona_eur')]['limite_inf']
outliers_precio = con.execute(f'''
    SELECT tour_id, descripcion, precio_por_persona_eur, proveedor_id
    FROM tours
    WHERE precio_por_persona_eur > {hi_precio} OR precio_por_persona_eur < {lo_precio}
    ORDER BY precio_por_persona_eur DESC
''').df()
outliers_precio

,tour_id,descripcion,precio_por_persona_eur,proveedor_id
0,5057,Excursión de día completo desde Estambul,83.15,1011
1,5036,Excursión de día completo desde Londres,81.33,1003


### Conclusiones — outliers

- **`importe_eur`** (importe de la reserva)
La mitad de las reservas está entre 33€ y 133€ aproximadamente, con un valor central (mediana) de 74€. Por encima de 283€ ya se considera "extremo" según el cálculo estadístico, y ahí encontramos **237 reservas**, con la más cara en 415,75€.
    > No parecen errores: son importes coherentes con reservas de varias personas o tours más caros de lo normal. Conviene tratarlos como reservas grupales/premium legítimas, no como ruido a eliminar.

- **`personas`** (nº de personas en la reserva)
Aquí el cálculo estadístico no marca nada raro (el rango "normal" que calcula llega hasta -1, lo cual ya es una pista de que el método no aplica bien a este caso). Pero mirando los datos directamente, hay **30 reservas con 0 o menos personas** (15 con `0`, 15 con `-1`).
    > Esto es un error de datos claro, no un caso extremo: no puedes reservar un tour para "0 personas" o "-1 personas". Se debe corregir o eliminar, no depende de ningún cálculo estadístico.

- **`precio_por_persona_eur`** (catálogo de tours)
La mayoría de tours cuesta entre 17€ y 42€ por persona. Por encima de 80€ se considera extremo, y solo **2 tours** superan ese umbral (el más caro, 83,15€).
    > Desvío leve y esperable — probablemente tours premium o experiencias especiales.

- **`importe_eur` = 0€**
No hay importes negativos, pero sí **1.297 reservas con importe 0€**.
    > Se cruzó cada una de las 1.297 reservas con el `precio_por_persona_eur` de su tour en el catálogo (ver `/src/limpieza.py` y `reports/data_quality_report.md`). Resultado: **el 100% corresponde a tours cuyo precio de catálogo también es 0€** — son tours gratuitos legítimos, no cancelaciones sin cobro ni errores de captura. No hay ningún caso de tour de pago cobrado a 0€. Como son importes de 0€, no alteran `venta_bruta` ni `venta_neta`; se documenta con la columna `tour_gratuito` en `reservas_limpias` (sección de `limpieza.py`).
    >
    > Nota: 2 de esas 1.297 reservas también tienen `personas <= 0` y se eliminan en la limpieza (sección siguiente), así que en `reservas_limpias`/`reports/data_quality_report.md` la cifra de reservas con importe 0€ es **1.295**, no 1.297 — mismo hallazgo, medido antes y después de ese filtro.

## 4. Inconsistencias de fechas


In [20]:
inconsistencias_fechas = con.execute('''
    SELECT * FROM (
        SELECT 'clientes.fecha_baja < fecha_alta'       AS chequeo, count(*) AS n_casos FROM clientes    WHERE fecha_baja < fecha_alta
        UNION ALL
        SELECT 'proveedores.fecha_baja < fecha_alta',           count(*) FROM proveedores WHERE fecha_baja < fecha_alta
        UNION ALL
        SELECT 'reservas.fecha_actividad < fecha_reserva',      count(*) FROM reservas    WHERE fecha_actividad < CAST(fecha_reserva AS DATE)
        UNION ALL
        SELECT 'clientes.fecha_alta futura',                    count(*) FROM clientes    WHERE fecha_alta > CURRENT_DATE
        UNION ALL
        SELECT 'clientes.fecha_baja futura',                    count(*) FROM clientes    WHERE fecha_baja > CURRENT_DATE
        UNION ALL
        SELECT 'clientes.fecha_nacimiento futura',              count(*) FROM clientes    WHERE fecha_nacimiento > CURRENT_DATE
        UNION ALL
        SELECT 'proveedores.fecha_alta futura',                 count(*) FROM proveedores WHERE fecha_alta > CURRENT_DATE
        UNION ALL
        SELECT 'proveedores.fecha_baja futura',                 count(*) FROM proveedores WHERE fecha_baja > CURRENT_DATE
        UNION ALL
        SELECT 'reservas.fecha_reserva futura',                 count(*) FROM reservas    WHERE CAST(fecha_reserva AS DATE) > CURRENT_DATE
        UNION ALL
        SELECT 'reservas.fecha_actividad futura',                count(*) FROM reservas    WHERE fecha_actividad > CURRENT_DATE
    )
    ORDER BY n_casos DESC
''').df()
inconsistencias_fechas

,chequeo,n_casos
0,clientes.fecha_baja futura,39
1,reservas.fecha_actividad futura,21
2,clientes.fecha_alta futura,1
3,proveedores.fecha_alta futura,0
4,proveedores.fecha_baja futura,0
5,proveedores.fecha_baja < fecha_alta,0
6,clientes.fecha_baja < fecha_alta,0
7,reservas.fecha_reserva futura,0
8,reservas.fecha_actividad < fecha_reserva,0
9,clientes.fecha_nacimiento futura,0


### Verificación de `fecha_baja` y `fecha_alta` futuras

Los 39 casos de `clientes.fecha_baja` futura y el caso de `clientes.fecha_alta` futura de la tabla anterior podrían tener una explicación operativa, no ser un error de captura: que el cliente programase su baja para justo después de disfrutar su última reserva, o que su alta coincidiera con el momento de su primera reserva. Se comprueba aquí si esa hipótesis se sostiene con los datos.

In [21]:
bajas_futuras = con.execute('''
    SELECT
        c.user_id,
        c.fecha_baja,
        max(r.fecha_actividad)                                     AS ultima_fecha_actividad,
        date_diff('day', max(r.fecha_actividad), c.fecha_baja)     AS dif_dias
    FROM clientes c
    LEFT JOIN reservas r ON r.user_id = c.user_id
    WHERE c.fecha_baja > CURRENT_DATE
    GROUP BY c.user_id, c.fecha_baja
    ORDER BY dif_dias
''').df()

MARGEN_DIAS = 3
sin_reservas = bajas_futuras['ultima_fecha_actividad'].isna().sum()
coincide_exacto = (bajas_futuras['dif_dias'] == 0).sum()
coincide_margen = (bajas_futuras['dif_dias'].abs() <= MARGEN_DIAS).sum()

print(f"Nº de clientes con fecha_baja futura: {len(bajas_futuras)}")
print(f"  - sin ninguna reserva registrada: {sin_reservas}")
print(f"  - fecha_baja coincide EXACTO con su última fecha_actividad: {coincide_exacto}")
print(f"  - fecha_baja coincide dentro de un margen de {MARGEN_DIAS} días: {coincide_margen}")
bajas_futuras

Nº de clientes con fecha_baja futura: 39
  - sin ninguna reserva registrada: 8
  - fecha_baja coincide EXACTO con su última fecha_actividad: 0
  - fecha_baja coincide dentro de un margen de 3 días: 0


,user_id,fecha_baja,ultima_fecha_actividad,dif_dias
0,23294,2027-01-08,2026-07-10,182
1,26660,2026-11-11,2026-05-10,185
2,27969,2027-01-23,2026-07-08,199
3,22916,2027-02-11,2026-06-28,228
4,26655,2027-05-17,2026-08-06,284
5,23721,2027-06-01,2026-08-01,304
6,23853,2027-03-13,2026-03-16,362
7,25756,2027-06-01,2026-06-04,362
8,27798,2026-11-02,2025-10-23,375
9,23555,2026-09-19,2025-09-01,383


**Resultado: 0 de los 39 casos coincide** con la fecha de su última reserva, ni de forma exacta ni dentro de un margen de 3 días — la diferencia mínima observada es de 182 días, muy lejos de un patrón de "doy de baja justo tras el último tour". Además, 8 de los 39 clientes no tienen ninguna reserva registrada, así que en esos casos ni siquiera hay una `fecha_actividad` con la que comparar.Esto descarta la hipótesis de que la `fecha_baja` futura esté ligada al fin de una reserva.

In [22]:
alta_futura = con.execute('''
    SELECT
        c.user_id,
        c.fecha_alta,
        min(CAST(r.fecha_reserva AS DATE))  AS primera_fecha_reserva,
        min(r.fecha_actividad)              AS primera_fecha_actividad
    FROM clientes c
    LEFT JOIN reservas r ON r.user_id = c.user_id
    WHERE c.fecha_alta > CURRENT_DATE
    GROUP BY c.user_id, c.fecha_alta
''').df()
alta_futura

,user_id,fecha_alta,primera_fecha_reserva,primera_fecha_actividad
0,28043,2027-01-11,NaT,NaT


El único cliente (`user_id` 28043) con `fecha_alta` futura (2027-01-11) **no tiene ninguna reserva asociada** — `primera_fecha_reserva` y `primera_fecha_actividad` salen ambas vacías. No hay, por tanto, ninguna reserva con la que su alta pudiera coincidir, así que tampoco se puede confirmar aquí la hipótesis de un alta "programada" junto a una primera compra.

`ga_eventos.event_date` es `VARCHAR` porque mezcla formatos
(`YYYY-MM-DD HH:MI:SS` y `DD/MM/YYYY HH:MI`). Se cuantifica cuántas filas no
son parseables con el formato ISO para dimensionar el problema:

In [23]:
con.execute('''
    SELECT
        count(*)                                                              AS total_filas,
        sum(CASE WHEN try_cast(event_date AS TIMESTAMP) IS NULL THEN 1 ELSE 0 END) AS no_parseable_iso,
        min(try_cast(event_date AS TIMESTAMP))                                AS fecha_min_parseada,
        max(try_cast(event_date AS TIMESTAMP))                                AS fecha_max_parseada
    FROM ga_eventos
''').df()

,total_filas,no_parseable_iso,fecha_min_parseada,fecha_max_parseada
0,702821,32589.0,2024-06-30 09:50:55,2026-06-30 23:51:21


### Conclusiones — fechas

- **Comprobaciones de orden lógico**

    - Nadie tiene una fecha de baja anterior a su fecha de alta — 0 casos en `clientes` y en `proveedores`.

    - Nadie reserva una actividad para una fecha ya pasada respecto al momento de la reserva — 0 casos en `reservas`.

- **Fechas que caen en el futuro** 

    | Dónde | Casos | ¿Es un error? |
    |---|---|---|
    | `clientes.fecha_baja` | 39 | Sí, ya que no asumimos que puedas darte de baja en una fecha futura  |
    | `clientes.fecha_alta` | 1 | "" de alta ""|
    | `reservas.fecha_actividad` | 24 | No — probablemente sean tours reservados para próximas fechas |

    El resto de fechas del dataset (nacimiento de clientes, alta/baja de proveedores, fecha de reserva) no presentan ningún caso futuro.

    Se comprobó si esas fechas de alta/baja futuras respondían a un patrón operativo (baja programada justo tras el último tour, alta programada junto a la primera reserva) — **no es el caso en ninguno de los 40 clientes** (0/39 en `fecha_baja`, y el único caso de `fecha_alta` no tiene ninguna reserva asociada). Detalle en la verificación de las celdas anteriores.

- **Formato de fecha mezclado en `ga_eventos.event_date`**

    32.589 filas (~4,6%) no se pudieron leer como fecha. La causa: la columna mezcla dos formatos distintos — unas filas usan `DD/MM/YYYY HH:MI` (día/mes/año) y otras `YYYY-MM-DD HH:MI:SS` (año-mes-día). El sistema esperaba solo el segundo formato, así que todo lo que llegó en el primero no se pudo interpretar como fecha.  

## 5. Resumen de conclusiones

**Duplicados y formato**
- `ga_eventos` tiene 798 filas duplicadas exactas (el mismo evento registrado dos veces).
- `ga_eventos.event_date` mezcla dos formatos de fecha distintos: 32.589 filas (~4,6%) no se pudieron leer como fecha.
- `reservas.estado` y `ga_eventos.device` mezclan mayúsculas/minúsculas y un typo para el mismo valor (detalle en sección 1).

**Relaciones rotas entre ficheros**
- 2.238 de los 8.060 clientes (~28%) están registrados pero nunca han hecho ninguna reserva.
- 40 de 8.341 eventos de compra (`purchase`) no tienen una reserva correspondiente en `reservas` (~0,5%).
- 187 reservas y 1 tour hacen referencia a un proveedor que no existe en el fichero de proveedores.

**Valores del importe de reserva fuera de lo normal**
- 237 reservas superan el límite estadístico de "importe normal" (283,22€), con un máximo de 415,75€ — no parecen errores, más bien reservas grupales o premium.
- 1.297 reservas tienen importe de 0€: se cruzaron con el precio de catálogo de su tour y el 100% corresponde a tours cuyo `precio_por_persona_eur` también es 0€ (tours gratuitos legítimos, no cancelaciones sin cobro). Detalle en la sección de outliers más arriba y en `reports/data_quality_report.md`.
- 2 tours del catálogo tienen un precio por persona muy por encima del resto (máximo 83,15€) — desvío leve, esperable.

**Número de personas de la reserva incorrecto**
- 30 reservas tienen registradas 0 o menos personas (15 con el valor `0`, 15 con `-1`) — esto es un error de captura de datos, no una reserva legítima con pocos asistentes.

**Fechas en el futuro**
- 39 clientes tienen una fecha de baja posterior a hoy — se comprobó si esa fecha coincidía con la fecha de su última reserva (baja programada justo tras el último tour) y **no es el caso en ninguno de los 39** (8 de ellos ni siquiera tienen reservas).
- 1 cliente tiene una fecha de alta posterior a hoy — tampoco coincide con ninguna reserva, porque ese cliente no tiene ninguna registrada.
- 24 reservas tienen `fecha_actividad` en el futuro — esto es esperable (tours ya reservados que aún no se han realizado), no se considera un error.


Nótese que ninguno de estos hallazgos se ha corregido en este notebook — quedan
documentados aquí para diseñar la lógica de limpieza en `/src` en un paso
posterior. La única excepción es la decisión de `importe_eur = 0€`, que
quedó pendiente en la primera versión de este notebook y se ha resuelto
investigando con SQL sobre los datos — el tratamiento resultante ya está
implementado en `/src/limpieza.py` (columna `tour_gratuito`).

### Hallazgos detectados pero no corregidos

Tres de los problemas detectados en este notebook se documentan aquí, pero deliberadamente NO se corrigen en la limpieza posterior (`/src/limpieza.py`). Se explica el motivo de cada uno, con evidencia que apoya la decisión, y se deja como recomendación futura que alguien de negocio en Civitatis confirme si el criterio aplicado es correcto.

**1. `event_date` con formato mixto (32.589 filas, ~4,6%, no parseables)**: no se corrige porque ninguna métrica del proyecto depende de esta columna. Se documenta el problema, pero no se invierte tiempo en parsearlo al no tener impacto en ninguna conclusión ya que `event_date` solo se usa para cosas relacionadas con comportamiento web/sesiones. 

**2. `purchase` sin `reserva_id` correspondiente en `reservas` (40 eventos)**: para valorar si son compras reales con un desfase de sincronización, o eventos corruptos/de prueba, se comprueba si el resto de columnas de esos 40 eventos está relleno.

In [24]:
eventos_purchase_sin_reserva = con.execute('''
    SELECT
        count(*)                                                                       AS total_eventos,
        sum(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END)                               AS user_id_nulos,
        sum(CASE WHEN cookie_id IS NULL OR cookie_id = '' THEN 1 ELSE 0 END)            AS cookie_id_nulos,
        sum(CASE WHEN temp_client_id IS NULL OR temp_client_id = '' THEN 1 ELSE 0 END)  AS temp_client_id_nulos,
        sum(CASE WHEN session_id IS NULL OR session_id = '' THEN 1 ELSE 0 END)          AS session_id_nulos,
        sum(CASE WHEN event_date IS NULL OR event_date = '' THEN 1 ELSE 0 END)          AS event_date_nulos,
        sum(CASE WHEN device IS NULL OR device = '' THEN 1 ELSE 0 END)                  AS device_nulos,
        sum(CASE WHEN ip IS NULL OR ip = '' THEN 1 ELSE 0 END)                          AS ip_nulos,
        sum(CASE WHEN pais_ip IS NULL OR pais_ip = '' THEN 1 ELSE 0 END)                AS pais_ip_nulos,
        sum(CASE WHEN ciudad_ip IS NULL OR ciudad_ip = '' THEN 1 ELSE 0 END)            AS ciudad_ip_nulos,
        sum(CASE WHEN url IS NULL OR url = '' THEN 1 ELSE 0 END)                        AS url_nulos
    FROM ga_eventos g
    WHERE g.reserva_id IS NOT NULL
      AND NOT EXISTS (SELECT 1 FROM reservas r WHERE r.reserva_id = g.reserva_id)
''').df()
print(f"Nº de eventos purchase sin reserva_id correspondiente: {eventos_purchase_sin_reserva['total_eventos'][0]}")
eventos_purchase_sin_reserva

Nº de eventos purchase sin reserva_id correspondiente: 40


,total_eventos,user_id_nulos,cookie_id_nulos,temp_client_id_nulos,session_id_nulos,event_date_nulos,device_nulos,ip_nulos,pais_ip_nulos,ciudad_ip_nulos,url_nulos
0,40,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Estos 40 eventos tienen el resto de sus campos rellenos — `user_id`, `cookie_id`, `temp_client_id`, `session_id`, `event_date`, `device`, `ip`, `pais_ip`, `ciudad_ip` y `url` no presentan ningún nulo en ninguno de los 40 casos —, lo que apoya la hipótesis de que son compras reales con un desfase de sincronización entre analítica y base de datos, no eventos corruptos o de prueba. Por eso no se descartan del análisis, aunque tampoco se puede confirmar con certeza sin acceso al sistema de origen. Aún así, suponen un porcentaje pequeño (~0.5). 

**Recomendación futura**: confirmar con el equipo técnico que gestiona el guardado de reservas si existe un patrón conocido de fallos o reintentos en ese proceso.

**3. `proveedor_id` inexistente en `proveedores` (187 reservas + 1 tour)**: para valorar si son reservas legítimas con solo el dato de proveedor roto, o si tienen más señales de estar corruptas, se comprueba el resto de columnas de esas 187 reservas.

In [25]:
reservas_sin_proveedor_valido = con.execute('''
    SELECT
        count(*)                                                                                              AS total_reservas,
        sum(CASE WHEN r.user_id IS NULL THEN 1 ELSE 0 END)                                                    AS user_id_nulos,
        sum(CASE WHEN NOT EXISTS (SELECT 1 FROM clientes c WHERE c.user_id = r.user_id) THEN 1 ELSE 0 END)     AS user_id_sin_cliente,
        sum(CASE WHEN r.tour_id IS NULL THEN 1 ELSE 0 END)                                                    AS tour_id_nulos,
        sum(CASE WHEN NOT EXISTS (SELECT 1 FROM tours t WHERE t.tour_id = r.tour_id) THEN 1 ELSE 0 END)       AS tour_id_sin_tour,
        sum(CASE WHEN r.importe_eur IS NULL THEN 1 ELSE 0 END)                                                AS importe_eur_nulos,
        sum(CASE WHEN r.fecha_reserva IS NULL THEN 1 ELSE 0 END)                                              AS fecha_reserva_nulos,
        sum(CASE WHEN r.fecha_actividad IS NULL THEN 1 ELSE 0 END)                                            AS fecha_actividad_nulos,
        sum(CASE WHEN r.estado IS NULL OR r.estado = '' THEN 1 ELSE 0 END)                                    AS estado_nulos,
        sum(CASE WHEN r.personas IS NULL THEN 1 ELSE 0 END)                                                   AS personas_nulos
    FROM reservas r
    WHERE NOT EXISTS (SELECT 1 FROM proveedores p WHERE p.proveedor_id = r.proveedor_id)
''').df()
print(f"Nº de reservas sin proveedor_id válido: {reservas_sin_proveedor_valido['total_reservas'][0]}")
reservas_sin_proveedor_valido

Nº de reservas sin proveedor_id válido: 187


,total_reservas,user_id_nulos,user_id_sin_cliente,tour_id_nulos,tour_id_sin_tour,importe_eur_nulos,fecha_reserva_nulos,fecha_actividad_nulos,estado_nulos,personas_nulos
0,187,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Estas 187 reservas tienen el resto de sus campos rellenos y coherentes: `user_id`, `tour_id`, `importe_eur`, `fecha_reserva`, `fecha_actividad`, `estado` y `personas` no presentan ningún nulo, y tanto `user_id` como `tour_id` referencian siempre un cliente y un tour existentes (0 casos sin match). Esto apoya que son reservas reales con un problema aislado en el campo `proveedor_id` (posible baja de proveedor sin actualizar la referencia, o error de captura puntual), no reservas corruptas en su conjunto. No se excluyen del análisis, ya que ninguna consulta de negocio en `/src/queries.py` necesita cruzar con `proveedores`. 

**Recomendación futura**: confirmar con el equipo de proveedores/operaciones si estos `proveedor_id` corresponden a proveedores dados de baja del catálogo, para decidir si conviene reconciliarlos.